# FSL 15-Sign Trainer — Conv1D + LSTM

**Dataset:** [previlace/fsl-105](https://www.kaggle.com/datasets/previlace/fsl-105) — `labels.csv` + `fsl_clips/<id>/*.MOV`

**Landmark extraction** samples **30 frames/video** (not every frame), shows a **tqdm** progress bar, and saves a **partial checkpoint** every 10 videos so you can resume if Colab disconnects.

**Quick test:** set `MAX_VIDEOS_PER_SIGN = 5` in the data-loading cell.

In [ ]:
!pip -q install mediapipe opencv-python-headless scikit-learn joblib tensorflow pandas tqdm

In [ ]:
import os, shutil, urllib.request, zipfile
from pathlib import Path

from google.colab import userdata

os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

!pip -q install kaggle

# https://www.kaggle.com/datasets/previlace/fsl-105
KAGGLE_DATASET = 'previlace/fsl-105'
ZIP_PATH = Path('/content/fsl-105.zip')
DATA_ROOT = Path('/content/fsl_dataset')  # resolved after download

CHECKPOINT_DIR = Path('/content/checkpoints/fsl_15_lstm')
RAW_CHECKPOINT = CHECKPOINT_DIR / '01_raw_landmarks.npz'
PREPROCESSED_CHECKPOINT = CHECKPOINT_DIR / '02_preprocessed_augmented.npz'
TRAINING_DIR = CHECKPOINT_DIR / 'training'
for p in (CHECKPOINT_DIR, TRAINING_DIR):
    p.mkdir(parents=True, exist_ok=True)

USE_DRIVE = False  # set True to persist checkpoints on Google Drive
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    CHECKPOINT_DIR = Path('/content/drive/MyDrive/fsl_15_checkpoints/lstm')
    RAW_CHECKPOINT = CHECKPOINT_DIR / '01_raw_landmarks.npz'
    PREPROCESSED_CHECKPOINT = CHECKPOINT_DIR / '02_preprocessed_augmented.npz'
    TRAINING_DIR = CHECKPOINT_DIR / 'training'
    CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
    TRAINING_DIR.mkdir(parents=True, exist_ok=True)


def resolve_dataset_root(search_root: Path) -> Path:
    """Find folder containing labels.csv (dataset root)."""
    if (search_root / 'labels.csv').exists():
        return search_root
    matches = list(search_root.rglob('labels.csv'))
    if matches:
        return matches[0].parent
    return search_root


def extract_nested_clips_zip(dataset_root: Path):
    """Extract fsl_clips.zip or clips.zip if videos are not already on disk."""
    videos = list(dataset_root.rglob('*.mov')) + list(dataset_root.rglob('*.mp4'))
    videos += list(dataset_root.rglob('*.MOV')) + list(dataset_root.rglob('*.MP4'))
    if len(videos) > 100:
        return

    for zip_name in ('fsl_clips.zip', 'clips.zip'):
        clips_zip = dataset_root / zip_name
        if not clips_zip.exists():
            continue
        out_name = zip_name.replace('.zip', '')
        out_dir = dataset_root / out_name
        out_dir.mkdir(parents=True, exist_ok=True)
        print(f'Extracting {zip_name} (may take several minutes)...')
        with zipfile.ZipFile(clips_zip, 'r') as zf:
            zf.extractall(out_dir)


def download_fsl105() -> Path:
    """Download previlace/fsl-105 — labels.csv + fsl_clips/<id>/*.MOV"""
    global DATA_ROOT

    existing = resolve_dataset_root(Path('/content/fsl_dataset'))
    if (existing / 'labels.csv').exists():
        videos = list(existing.rglob('*.mov')) + list(existing.rglob('*.MOV'))
        videos += list(existing.rglob('*.mp4')) + list(existing.rglob('*.MP4'))
        if len(videos) > 100:
            DATA_ROOT = existing
            print(f'Dataset ready at {DATA_ROOT} ({len(videos)} videos)')
            return DATA_ROOT

    print(f'Downloading {KAGGLE_DATASET}...')
    os.system(f'kaggle datasets download -d {KAGGLE_DATASET} -p /content')
    if not ZIP_PATH.exists():
        raise FileNotFoundError(f'Expected {ZIP_PATH} after kaggle download')

    extract_to = Path('/content/fsl_dataset')
    extract_to.mkdir(parents=True, exist_ok=True)
    print('Extracting fsl-105.zip...')
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall(extract_to)

    DATA_ROOT = resolve_dataset_root(extract_to)
    extract_nested_clips_zip(DATA_ROOT)

    labels_csv = DATA_ROOT / 'labels.csv'
    videos = list(DATA_ROOT.rglob('*.mov')) + list(DATA_ROOT.rglob('*.MOV'))
    videos += list(DATA_ROOT.rglob('*.mp4')) + list(DATA_ROOT.rglob('*.MP4'))

    print('Dataset root:', DATA_ROOT)
    print('labels.csv:', labels_csv.exists(), labels_csv if labels_csv.exists() else '')
    print('train.csv:', (DATA_ROOT / 'train.csv').exists())
    print('Video clips found:', len(videos))

    if not labels_csv.exists():
        raise FileNotFoundError('labels.csv not found after extract')
    if len(videos) == 0:
        print('Top-level contents:')
        for p in sorted(DATA_ROOT.iterdir())[:20]:
            print(' ', p)
    return DATA_ROOT


RAW_PARTIAL = CHECKPOINT_DIR / '01_raw_landmarks.partial.npz'

DATA_ROOT = download_fsl105()
print('Checkpoint dir:', CHECKPOINT_DIR)
print('Partial checkpoint:', RAW_PARTIAL)

In [ ]:
import numpy as np

# 15 curated labels (match live app convention)
ACTIONS = np.array([
    'GOOD_AFTERNOON', 'NICE_TO_MEET_YOU', 'YES', 'NO', 'THANK_YOU',
    'HOW_ARE_YOU', 'ONE', 'TWO', 'THREE', 'SIX', 'SEVEN',
    'CORRECT', 'WRONG', 'IM_FINE', 'UNDERSTAND',
])
SEQUENCE_LENGTH = 30
KEYPOINT_DIM = 258
NUM_CLASSES = len(ACTIONS)

# Folder name aliases if Kaggle uses different naming
LABEL_ALIASES = {
    'good afternoon': 'GOOD_AFTERNOON',
    'good_afternoon': 'GOOD_AFTERNOON',
    'nice to meet you': 'NICE_TO_MEET_YOU',
    'nice_to_meet_you': 'NICE_TO_MEET_YOU',
    'thank you': 'THANK_YOU',
    'thank_you': 'THANK_YOU',
    'how are you': 'HOW_ARE_YOU',
    'how_are_you': 'HOW_ARE_YOU',
    "i'm fine": 'IM_FINE',
    'im fine': 'IM_FINE',
    'im_fine': 'IM_FINE',
}

def normalize_label(name: str) -> str | None:
    """Map labels.csv text (e.g. 'IM FINE', 'GOOD AFTERNOON') to ACTION ids."""
    key = name.strip().replace("'", '').replace('-', ' ')
    upper = '_'.join(key.upper().split())
    if upper in ACTIONS:
        return upper
    lower = key.lower()
    if lower in LABEL_ALIASES:
        return LABEL_ALIASES[lower]
    compact = lower.replace(' ', '_')
    if compact in LABEL_ALIASES:
        return LABEL_ALIASES[compact]
    return None

print('Training labels:', list(ACTIONS))

## Landmark extraction (MediaPipe Tasks — HolisticLandmarker)

In [ ]:
import cv2
import mediapipe as mp
from mediapipe.tasks.python.core import base_options as base_options_module
from mediapipe.tasks.python.vision import HolisticLandmarker, HolisticLandmarkerOptions, RunningMode

MODEL_PATH = Path('/content/holistic_landmarker.task')
if not MODEL_PATH.exists():
    url = 'https://storage.googleapis.com/mediapipe-models/holistic_landmarker/holistic_landmarker/float16/latest/holistic_landmarker.task'
    urllib.request.urlretrieve(url, MODEL_PATH)

options = HolisticLandmarkerOptions(
    base_options=base_options_module.BaseOptions(model_asset_path=str(MODEL_PATH)),
    running_mode=RunningMode.VIDEO,
    min_pose_detection_confidence=0.5,
    min_pose_landmarks_confidence=0.5,
    min_hand_landmarks_confidence=0.5,
)
landmarker = HolisticLandmarker.create_from_options(options)

# Monotonic across all videos (required for VIDEO mode)
_TIMESTAMP_MS = 0


def extract_keypoints(result):
    pose = np.zeros(33 * 4, dtype=np.float32)
    if result.pose_landmarks:
        for i, lm in enumerate(result.pose_landmarks):
            pose[i*4:(i+1)*4] = [lm.x, lm.y, lm.z, getattr(lm, 'visibility', 0.0)]
    lh = np.zeros(21 * 3, dtype=np.float32)
    if result.left_hand_landmarks:
        for i, lm in enumerate(result.left_hand_landmarks):
            lh[i*3:(i+1)*3] = [lm.x, lm.y, lm.z]
    rh = np.zeros(21 * 3, dtype=np.float32)
    if result.right_hand_landmarks:
        for i, lm in enumerate(result.right_hand_landmarks):
            rh[i*3:(i+1)*3] = [lm.x, lm.y, lm.z]
    return np.concatenate([pose, lh, rh])


def video_to_sequence(video_path, target_len=30):
    """Extract exactly target_len frames — MediaPipe runs on sampled frames only."""
    global _TIMESTAMP_MS
    cap = cv2.VideoCapture(str(video_path))
    bgr_frames = []
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        bgr_frames.append(frame)
    cap.release()
    if not bgr_frames:
        return None

    n = len(bgr_frames)
    pick = np.linspace(0, n - 1, min(target_len, n)).astype(int)
    keypoints = []
    for idx in pick:
        rgb = cv2.cvtColor(bgr_frames[idx], cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        _TIMESTAMP_MS += 33
        result = landmarker.detect_for_video(mp_image, _TIMESTAMP_MS)
        keypoints.append(extract_keypoints(result))

    arr = np.array(keypoints, dtype=np.float32)
    if len(arr) < target_len:
        pad = np.repeat(arr[-1:], target_len - len(arr), axis=0)
        arr = np.vstack([arr, pad])
    return arr


def set_timestamp_ms(value):
    global _TIMESTAMP_MS
    _TIMESTAMP_MS = int(value)

print('HolisticLandmarker ready (VIDEO mode, 30 frames/video max)')

In [ ]:
import pandas as pd
from tqdm.auto import tqdm

# Set to an int to cap clips/sign for a quick test (None = use all clips)
MAX_VIDEOS_PER_SIGN = None
SAVE_EVERY_N_VIDEOS = 10


def save_partial_checkpoint(sequences, labels, processed_paths, timestamp_ms):
    np.savez_compressed(
        RAW_PARTIAL,
        sequences=np.array(sequences, dtype=object),
        labels=np.array(labels, dtype=int),
        processed_paths=np.array(sorted(processed_paths)),
        timestamp_ms=np.array([timestamp_ms], dtype=np.int64),
        actions=ACTIONS,
    )


def load_partial_checkpoint():
    if not RAW_PARTIAL.exists():
        return [], [], set(), 0
    data = np.load(RAW_PARTIAL, allow_pickle=True)
    ts = int(data['timestamp_ms'][0]) if 'timestamp_ms' in data else 0
    paths = set(str(p) for p in data['processed_paths'])
    print(f'Resuming partial checkpoint: {len(paths)} videos already processed')
    return list(data['sequences']), list(data['labels']), paths, ts
    """Locate fsl_clips/<id>/*.MOV — previlace/fsl-105 layout."""
    candidates = [
        fsl_root / 'fsl_clips',
        fsl_root / 'clips',
        fsl_root / 'fsl_clips' / 'fsl_clips',
        fsl_root / 'clips' / 'clips',
    ]
    for root in candidates:
        if not root.exists():
            continue
        numeric = [p for p in root.iterdir() if p.is_dir() and p.name.isdigit()]
        if len(numeric) >= 5:
            return root
    best, best_count = None, 0
    for parent in fsl_root.rglob('*'):
        if not parent.is_dir() or parent.name.startswith('.'):
            continue
        numeric = [p for p in parent.iterdir() if p.is_dir() and p.name.isdigit()]
        if len(numeric) > best_count:
            best, best_count = parent, len(numeric)
    return best if best_count >= 5 else None


def find_clips_root(fsl_root: Path):
    """Locate fsl_clips/<id>/*.MOV — previlace/fsl-105 layout."""
    candidates = [
        fsl_root / 'fsl_clips',
        fsl_root / 'clips',
        fsl_root / 'fsl_clips' / 'fsl_clips',
        fsl_root / 'clips' / 'clips',
    ]
    for root in candidates:
        if not root.exists():
            continue
        numeric = [p for p in root.iterdir() if p.is_dir() and p.name.isdigit()]
        if len(numeric) >= 5:
            return root
    best, best_count = None, 0
    for parent in fsl_root.rglob('*'):
        if not parent.is_dir() or parent.name.startswith('.'):
            continue
        numeric = [p for p in parent.iterdir() if p.is_dir() and p.name.isdigit()]
        if len(numeric) > best_count:
            best, best_count = parent, len(numeric)
    return best if best_count >= 5 else None


def load_id_to_action(fsl_root: Path):
    """Read labels.csv: id -> normalized action name."""
    labels_csv = fsl_root / 'labels.csv'
    if not labels_csv.exists():
        labels_csv = next(iter(fsl_root.rglob('labels.csv')), None)
    if labels_csv is None:
        return {}, None

    df = pd.read_csv(labels_csv)
    print('labels.csv columns:', list(df.columns))
    print(df.head(12).to_string(index=False))

    cols = {c.lower(): c for c in df.columns}
    id_col = cols.get('id') or df.columns[0]
    label_col = cols.get('label') or cols.get('gloss') or cols.get('name') or df.columns[1]

    mapping = {}
    for _, row in df.iterrows():
        try:
            clip_id = int(row[id_col])
        except (ValueError, TypeError):
            continue
        action = normalize_label(str(row[label_col]))
        if action:
            mapping[clip_id] = action

    print(f'\nMapped {len(mapping)} ids from labels.csv')
    print('Our 15 target signs in catalog:')
    for clip_id, action in sorted(mapping.items()):
        if action in ACTIONS:
            print(f'  id {clip_id:3d} -> {action}')
    return mapping, labels_csv


def load_from_fsl105(fsl_root: Path):
    """Load clips for 15 signs with progress bar + resumable partial saves."""
    sequences, labels, processed_paths, ts = load_partial_checkpoint()
    set_timestamp_ms(ts)

    video_exts = {'.mp4', '.avi', '.mov', '.mkv', '.MOV', '.MP4', '.AVI', '.MKV'}
    clips_root = find_clips_root(fsl_root)
    if clips_root is None:
        raise FileNotFoundError('Could not find fsl_clips/ — re-run download cell')
    print('Clips root:', clips_root)

    id_to_action, _ = load_id_to_action(fsl_root)
    if not id_to_action:
        raise FileNotFoundError('labels.csv missing or empty')

    # Collect work list first so tqdm shows total
    jobs = []
    for clip_id, action in sorted(id_to_action.items()):
        if action not in ACTIONS:
            continue
        folder = clips_root / str(clip_id)
        if not folder.is_dir():
            continue
        videos = sorted([f for f in folder.iterdir() if f.is_file() and f.suffix in video_exts])
        if MAX_VIDEOS_PER_SIGN is not None:
            videos = videos[:MAX_VIDEOS_PER_SIGN]
        label_idx = int(np.where(ACTIONS == action)[0][0])
        for vid in videos:
            key = str(vid.resolve())
            if key not in processed_paths:
                jobs.append((vid, label_idx, action, key))

    print(f'Videos to process: {len(jobs)} ({len(processed_paths)} already done)')
    sign_counts = {a: 0 for a in ACTIONS}
    for seq, lab in zip(sequences, labels):
        sign_counts[ACTIONS[lab]] += 1

    since_save = 0
    for vid, label_idx, action, key in tqdm(jobs, desc='Extracting landmarks', unit='video'):
        seq = video_to_sequence(vid)
        if seq is not None:
            sequences.append(seq)
            labels.append(label_idx)
            sign_counts[action] += 1
        processed_paths.add(key)
        since_save += 1
        if since_save >= SAVE_EVERY_N_VIDEOS:
            save_partial_checkpoint(sequences, labels, processed_paths, _TIMESTAMP_MS)
            since_save = 0

    save_partial_checkpoint(sequences, labels, processed_paths, _TIMESTAMP_MS)

    print('\nClips loaded per sign:')
    total = sum(sign_counts.values())
    for action in ACTIONS:
        n = sign_counts[action]
        print(f'  {action:20s} {n:3d}  [{"ok" if n else "MISSING"}]')
    print(f'Total sequences: {total}')
    return sequences, labels


def find_data_roots(base: Path):
    """Locate FSL-105 Kaggle layout, MP_Data, or generic video folders."""
    if base.exists() and (next(base.rglob('labels.csv'), None) or find_clips_root(base)):
        return 'fsl105', base
    candidates = list(base.rglob('MP_Data'))
    if candidates:
        return 'mp_data', candidates[0]
    if base.exists() and any(base.iterdir()):
        return 'videos', base
    return None, None


def load_from_mp_data(mp_root: Path):
    sequences, labels = [], []
    for folder in mp_root.iterdir():
        if not folder.is_dir():
            continue
        label = normalize_label(folder.name)
        if label is None:
            continue
        label_idx = int(np.where(ACTIONS == label)[0][0])
        for seq_dir in folder.iterdir():
            if not seq_dir.is_dir():
                continue
            window = []
            for f in range(SEQUENCE_LENGTH):
                fp = seq_dir / f'{f}.npy'
                if not fp.exists():
                    break
                window.append(np.load(fp))
            if len(window) == SEQUENCE_LENGTH:
                sequences.append(window)
                labels.append(label_idx)
    return sequences, labels
mode = 'fsl105'
if RAW_CHECKPOINT.exists():
    print(f'Loading finished raw checkpoint: {RAW_CHECKPOINT}')
    raw = np.load(RAW_CHECKPOINT, allow_pickle=True)
    sequences = list(raw['sequences'])
    labels = list(raw['labels'])
    mode = str(raw.get('mode', 'fsl105'))
else:
    mode, root = find_data_roots(DATA_ROOT)
    if mode == 'fsl105':
        sequences, labels = load_from_fsl105(root)
    elif mode == 'mp_data':
        sequences, labels = load_from_mp_data(root)
    else:
        sequences, labels = []

    if len(sequences) == 0:
        raise RuntimeError('No training data — check download + landmark cells')

    np.savez_compressed(
        RAW_CHECKPOINT,
        sequences=np.array(sequences, dtype=object),
        labels=np.array(labels),
        actions=ACTIONS,
        mode=mode,
    )
    if RAW_PARTIAL.exists():
        RAW_PARTIAL.unlink()
    print(f'Saved raw landmark checkpoint -> {RAW_CHECKPOINT}')

print(f'Mode: {mode}, sequences: {len(sequences)}')

## Live-matching augmentations

Mirrors real-time bone/landmark inference: tracking jitter, variable signing speed, partial hand dropout.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.utils import to_categorical
import joblib

SCALER_CHECKPOINT = CHECKPOINT_DIR / 'scaler.pkl'


def augment_sequence(seq, rng):
    """Augment a (30, 258) landmark sequence for live-like robustness."""
    out = seq.copy().astype(np.float32)

    noise = rng.normal(0, 0.015, size=(out.shape[0], out.shape[1]))
    noise[:, 2::3] *= 0.3
    out += noise

    if rng.random() < 0.5:
        src_len = rng.integers(22, 38)
        idx = np.linspace(0, len(out) - 1, src_len)
        stretched = np.array([np.interp(idx, np.arange(len(out)), out[:, j]) for j in range(out.shape[1])]).T
        new_idx = np.linspace(0, len(stretched) - 1, SEQUENCE_LENGTH)
        out = np.array([np.interp(new_idx, np.arange(len(stretched)), stretched[:, j]) for j in range(stretched.shape[1])]).T

    pose_end = 33 * 4
    lh_end = pose_end + 21 * 3
    if rng.random() < 0.25:
        out[:, pose_end:lh_end] = 0
    if rng.random() < 0.25:
        out[:, lh_end:] = 0

    scale = rng.uniform(0.92, 1.08)
    out[:, 0::3] *= scale
    out[:, 1::3] *= scale
    return out


if PREPROCESSED_CHECKPOINT.exists() and SCALER_CHECKPOINT.exists():
    print(f'Loading preprocessed checkpoint: {PREPROCESSED_CHECKPOINT}')
    prep = np.load(PREPROCESSED_CHECKPOINT)
    X_train = prep['X_train']
    X_test = prep['X_test']
    y_train = prep['y_train']
    y_test = prep['y_test']
    scaler = joblib.load(SCALER_CHECKPOINT)
    y_train_cat = to_categorical(y_train, num_classes=NUM_CLASSES)
    y_test_cat = to_categorical(y_test, num_classes=NUM_CLASSES)
    print('Train:', X_train.shape, 'Test:', X_test.shape, '(restored from checkpoint)')
else:
    rng = np.random.default_rng(42)
    X_base = np.array(sequences, dtype=np.float32)
    y = np.array(labels, dtype=int)

    AUG_PER_SAMPLE = 3
    X_aug, y_aug = [], []
    for seq, label in zip(X_base, y):
        X_aug.append(seq)
        y_aug.append(label)
        for _ in range(AUG_PER_SAMPLE):
            X_aug.append(augment_sequence(seq, rng))
            y_aug.append(label)

    X = np.array(X_aug, dtype=np.float32)
    y = np.array(y_aug, dtype=int)
    print('Augmented shape:', X.shape, 'labels:', y.shape)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.15, random_state=42, stratify=y,
    )

    scaler = StandardScaler()
    scaler.fit(X_train.reshape(-1, KEYPOINT_DIM))
    X_train = scaler.transform(X_train.reshape(-1, KEYPOINT_DIM)).reshape(X_train.shape)
    X_test = scaler.transform(X_test.reshape(-1, KEYPOINT_DIM)).reshape(X_test.shape)

    y_train_cat = to_categorical(y_train, num_classes=NUM_CLASSES)
    y_test_cat = to_categorical(y_test, num_classes=NUM_CLASSES)

    np.savez_compressed(
        PREPROCESSED_CHECKPOINT,
        X_train=X_train,
        X_test=X_test,
        y_train=y_train,
        y_test=y_test,
        actions=ACTIONS,
    )
    joblib.dump(scaler, SCALER_CHECKPOINT)
    print(f'Saved preprocessed checkpoint -> {PREPROCESSED_CHECKPOINT}')
    print(f'Saved scaler -> {SCALER_CHECKPOINT}')
    print('Train:', X_train.shape, 'Test:', X_test.shape)

In [ ]:
# Preprocessed data loaded/saved in the cell above.
# Re-run that cell to rebuild from raw landmarks; delete 02_preprocessed_augmented.npz to force re-augmentation.
print('Ready for training:', X_train.shape, X_test.shape)

In [ ]:
import json
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Conv1D, LSTM, Dense, Dropout
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint, CSVLogger

BEST_MODEL_PATH = TRAINING_DIR / 'best_model.h5'
LATEST_MODEL_PATH = TRAINING_DIR / 'latest_model.h5'
HISTORY_PATH = TRAINING_DIR / 'training_history.json'

RESUME_TRAINING = BEST_MODEL_PATH.exists()

if RESUME_TRAINING:
    print(f'Resuming from checkpoint: {BEST_MODEL_PATH}')
    model = load_model(str(BEST_MODEL_PATH))
else:
    model = Sequential([
        Conv1D(64, kernel_size=3, activation='relu', input_shape=(SEQUENCE_LENGTH, KEYPOINT_DIM)),
        Dropout(0.3),
        LSTM(64, return_sequences=True, kernel_regularizer=l2(0.001)),
        Dropout(0.5),
        LSTM(128, return_sequences=False, kernel_regularizer=l2(0.001)),
        Dropout(0.5),
        Dense(64, activation='relu', kernel_regularizer=l2(0.001)),
        Dropout(0.3),
        Dense(NUM_CLASSES, activation='softmax'),
    ])
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

model.summary()

callbacks = [
    ModelCheckpoint(str(BEST_MODEL_PATH), monitor='val_accuracy', save_best_only=True, verbose=1),
    ModelCheckpoint(str(LATEST_MODEL_PATH), save_best_only=False, verbose=0),
    CSVLogger(str(TRAINING_DIR / 'training_log.csv'), append=RESUME_TRAINING),
    EarlyStopping(patience=25, restore_best_weights=True, monitor='val_accuracy'),
    ReduceLROnPlateau(factor=0.5, patience=10, min_lr=1e-6),
]

history = model.fit(
    X_train, y_train_cat,
    validation_data=(X_test, y_test_cat),
    epochs=200,
    batch_size=32,
    callbacks=callbacks,
)

# Reload best weights and persist training progress
model.load_weights(str(BEST_MODEL_PATH))
model.save(TRAINING_DIR / 'final_model.h5')
with open(HISTORY_PATH, 'w') as f:
    json.dump({k: [float(v) for v in vals] for k, vals in history.history.items()}, f, indent=2)

print(f'Saved best model  -> {BEST_MODEL_PATH}')
print(f'Saved final model -> {TRAINING_DIR / "final_model.h5"}')
print(f'Saved history     -> {HISTORY_PATH}')

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)
print('Test accuracy:', accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=ACTIONS))

In [ ]:
export_dir = Path('/content/fsl_15_export')
export_dir.mkdir(exist_ok=True)

model.save(export_dir / 'fsl_15_model.h5')
np.save(export_dir / 'action_labels_15.npy', ACTIONS)
joblib.dump(scaler, export_dir / 'scaler.pkl')

# Bundle all checkpoints for download
!cd {CHECKPOINT_DIR.parent} && zip -r /content/fsl_15_lstm_checkpoints.zip {CHECKPOINT_DIR.name}

print('Exported model  ->', export_dir)
print('All checkpoints ->', CHECKPOINT_DIR)
print('Checkpoint zip  -> /content/fsl_15_lstm_checkpoints.zip')
!ls -la /content/fsl_15_export
!ls -la {CHECKPOINT_DIR}
!ls -la {TRAINING_DIR}

In [ ]:
from google.colab import files

for fname in ['fsl_15_model.h5', 'action_labels_15.npy', 'scaler.pkl']:
    files.download(str(export_dir / fname))

files.download('/content/fsl_15_lstm_checkpoints.zip')